In [ ]:
%pip install google-generativeai mistralai openai pandas openpyxl tqdm --quiet

In [ ]:
import os, json, time
from concurrent.futures import ThreadPoolExecutor, as_completed
import pandas as pd
from tqdm import tqdm
import google.generativeai as genai
from mistralai import Mistral
from openai import OpenAI

print("Imports OK.")


In [ ]:

GEMINI_API_KEY   = "XXX"
MISTRAL_API_KEY  = "XXX"    
DEEPSEEK_API_KEY = "XXX"   


INPUT_FILE     = "DDL_Stage_Classification_v4.xlsx"
CHECKPOINT_CSV = "DDL_TE_Checkpoint.csv"
OUTPUT_EXCEL   = "DDL_TE_Scores.xlsx"


GEMINI_MODEL   = "gemini-2.5-flash-lite"
MISTRAL_MODEL  = "mistral-small-2506"
DEEPSEEK_MODEL = "deepseek-chat"


MAX_WORKERS = 3      
RETRY_DELAY = 10     
MAX_RETRIES = 3


genai.configure(api_key=GEMINI_API_KEY)
gemini_model    = genai.GenerativeModel(GEMINI_MODEL)
mistral_client  = Mistral(api_key=MISTRAL_API_KEY)
deepseek_client = OpenAI(api_key=DEEPSEEK_API_KEY, base_url="https://api.deepseek.com")

STAGE_NAMES = {
    1: "Stage 1 — Discovery & Development",
    2: "Stage 2 — Preclinical Research",
    3: "Stage 3 — Clinical Research",
    4: "Stage 4 — Regulatory Review",
}

print(f"Gemini   : {GEMINI_MODEL}")
print(f"Mistral  : {MISTRAL_MODEL}")
print(f"DeepSeek : {DEEPSEEK_MODEL}")
print(f"Workers  : {MAX_WORKERS} models in parallel per task")
print(f"Input    : {INPUT_FILE}")
print(f"Output   : {OUTPUT_EXCEL}")


In [ ]:
GENAI_SCOPE = (
    "Generative AI (GenAI) refers to foundation model-based systems — including Large Language "
    "Models (LLMs) and multimodal models — that coordinate multiple capabilities through a unified "
    "interface. These systems can process and generate text, code, images, and structured data.\n\n"
    "For this evaluation, assess TWO capability dimensions:\n"
    "  (i)  LLMs — textual understanding, generation, summarisation, structured reasoning, "
    "code interpretation, and knowledge retrieval.\n"
    "  (ii) Image Processing Systems — multimodal GenAI capabilities for analysing images, "
    "charts, microscopy outputs, scan results, laboratory data visualisations, and other "
    "visual inputs to support decision-making.\n\n"
    "IMPORTANT: Physical or robotic execution is NOT within scope. This evaluation focuses "
    "exclusively on cognitive, analytical, and information-processing GenAI capabilities, "
    "consistent with a focus on Generative AI rather than robotics."
)

RATING_SCALE = (
    "Rating scale (1 to 5):\n"
    "  1 = Poor     — GenAI can perform little to none of this task meaningfully. "
    "The task requires physical presence, hands-on manipulation, or capabilities far beyond "
    "current GenAI systems.\n"
    "  2 = Fair     — GenAI can assist with minor elements (e.g., documentation, "
    "background research) but the core task execution is beyond current GenAI capabilities.\n"
    "  3 = Average  — GenAI can perform a substantial and meaningful part of this task, "
    "but human domain expertise, physical access, or regulatory accountability remains essential "
    "for completion.\n"
    "  4 = Good     — GenAI can perform most of this task well. Human oversight is still "
    "needed for final validation, sign-off, or edge cases, but the cognitive core is highly "
    "automatable.\n"
    "  5 = Excellent — GenAI can perform this task at or near professional human level. "
    "The task is almost entirely cognitive and information-based, with minimal dependency "
    "on physical execution or contextual human judgement that GenAI cannot replicate."
)

FEW_SHOT_EXAMPLES = (
    "Five-shot examples (calibration):\n\n"

    "Occupation: Regulatory Affairs Specialists\n"
    "DDL Stage: Stage 4 — Regulatory Review\n"
    'Task: "Prepare and submit NDA dossier to the FDA, including clinical, preclinical, '
    'and manufacturing data sections."\n'
    '{"te_score": 4, "motivation": "LLMs can draft, structure, and cross-reference the '
    'full textual and tabular content of an NDA dossier with high proficiency. This includes '
    'summarising clinical study reports (CSRs), generating benefit-risk narratives, formatting '
    'regulatory correspondence, and ensuring consistency across modules. Image Processing Systems '
    'can extract and interpret embedded figures, tables, forest plots, and statistical charts from '
    'source documents. The primary limitation is that final regulatory submission requires human '
    'accountability, legal sign-off, and expert judgement on edge-case regulatory interpretation '
    '— functions that cannot be delegated to GenAI. Nevertheless, the cognitive and documentation-'
    'intensive core of NDA preparation is highly amenable to GenAI, justifying a score of 4."}\n\n'

    "Occupation: Clinical Research Coordinators\n"
    "DDL Stage: Stage 3 — Clinical Research\n"
    'Task: "Recruit patients for clinical trials by screening medical records and contacting '
    'eligible participants."\n'
    '{"te_score": 3, "motivation": "LLMs can support patient recruitment by screening '
    'unstructured clinical notes and records for inclusion/exclusion criteria, generating '
    'recruitment scripts and informational materials, and drafting personalised outreach '
    'communications. Image Processing Systems have limited applicability in the recruitment '
    'screening process itself. However, the relational and ethically sensitive nature of '
    'direct patient contact — obtaining informed consent, building trust, responding to '
    'concerns — requires human involvement that GenAI cannot replace. Additionally, direct '
    'EHR access is subject to privacy and regulatory constraints. GenAI therefore constitutes '
    'a meaningful augmentation tool for the information-processing aspects of recruitment but '
    'cannot perform the task end-to-end, justifying a score of 3."}\n\n'

    "Occupation: Biochemists and Biophysicists\n"
    "DDL Stage: Stage 1 — Discovery & Development\n"
    'Task: "Determine the three-dimensional structure of biological macromolecules using '
    'x-ray crystallography or cryo-electron microscopy."\n'
    '{"te_score": 4, "motivation": "Image Processing Systems are highly capable of analysing '
    'crystallographic diffraction patterns and cryo-EM density maps, and multimodal LLMs can '
    'interpret structural data outputs, generate structural hypotheses, and cross-reference '
    'results with databases such as PDB or UniProt. AlphaFold-class GenAI models have already '
    'demonstrated near-human performance in protein structure prediction from sequence data, '
    'substantially reducing the analytical burden of this task. LLMs can additionally assist '
    'with literature synthesis, data interpretation, and manuscript drafting around structural '
    'results. The remaining limitation is the physical laboratory work required to collect the '
    'diffraction or EM data itself, which falls outside GenAI scope. The analytical and '
    'interpretive core of this task, however, is highly amenable to GenAI, justifying a score of 4."}\n\n'

    "Occupation: Veterinary Assistants and Laboratory Animal Caretakers\n"
    "DDL Stage: Stage 2 — Preclinical Research\n"
    'Task: "Anesthetize and inoculate animals according to study protocol instructions."\n'
    '{"te_score": 1, "motivation": "This task is fundamentally a physical, in-vivo laboratory '
    'procedure requiring direct manual interaction with live animals — administering anaesthesia '
    'and injections under GLP-compliant conditions. Neither LLMs nor Image Processing Systems '
    'can physically perform these actions. LLMs could theoretically assist with protocol '
    'interpretation or procedural documentation, and Image Processing Systems might support '
    'post-procedure monitoring through image analysis, but these are peripheral contributions '
    'to a task whose core is entirely physical. As this evaluation excludes robotic execution, '
    'GenAI contribution to the actual performance of this task is negligible, warranting the '
    'lowest possible score of 1."}\n\n'

    "Occupation: Biostatisticians\n"
    "DDL Stage: Stage 3 — Clinical Research\n"
    'Task: "Calculate sample size requirements for clinical studies using statistical power '
    'and significance thresholds."\n'
    '{"te_score": 5, "motivation": "This task is entirely cognitive and computational. LLMs '
    'with advanced statistical reasoning capabilities can perform power calculations, select '
    'appropriate statistical frameworks (superiority, non-inferiority, adaptive designs), '
    'interpret input parameters (effect size, alpha level, desired power, dropout rate), '
    'and produce fully documented sample size estimates. They can generate reproducible code '
    'in R, Python, or SAS, and explain assumptions transparently. Image Processing Systems '
    'can support interpretation of published trial results used as the basis for effect size '
    'assumptions, for instance by extracting data from figures. There is no physical or '
    'relational component to this task, and the regulatory and statistical expertise required '
    'is well within the knowledge boundary of current frontier LLMs. This represents one of '
    'the highest-amenability tasks for GenAI in the DDL context, justifying a score of 5."}'
)

SYSTEM_PROMPT_TE = (
    "You are an expert in pharmaceutical drug development, occupational task analysis, "
    "and the technical capabilities of Generative AI systems.\n\n"
    "Your task is to evaluate how well current Generative AI could perform a specific occupational "
    "task that is part of the Drug Development Lifecycle (DDL).\n\n"
    + GENAI_SCOPE + "\n\n"
    + RATING_SCALE + "\n\n"
    + FEW_SHOT_EXAMPLES + "\n\n"
    "Respond ONLY with valid JSON, no other text:\n"
    '{"te_score": <integer 1-5>, "motivation": "<detailed paragraph that (1) explains which '
    'specific GenAI capabilities are or are not applicable to this task, (2) references both '
    'LLM and Image Processing dimensions where relevant, and (3) explicitly states why the '
    'score is not one point higher or lower>"}'
)

def build_te_prompt(occupation, stage_name, task):
    return (
        f'Now evaluate:\n'
        f'Occupation: "{occupation}"\n'
        f'DDL Stage: {stage_name}\n'
        f'Task: "{task}"'
    )

print(f"System prompt length: {len(SYSTEM_PROMPT_TE):,} characters")
print("Prompt ready.")


In [ ]:
def parse_te_response(raw):
    """Extract te_score (1-5) and motivation from LLM JSON response."""
    try:
        clean = raw.strip()
        if "```" in clean:
            parts = clean.split("```")
            clean = parts[1] if len(parts) > 1 else parts[0]
            if clean.startswith("json"):
                clean = clean[4:]
        start = clean.find("{")
        end   = clean.rfind("}") + 1
        if start != -1 and end > start:
            clean = clean[start:end]
        parsed = json.loads(clean.strip())
        score = int(parsed.get("te_score", -1))
        if score not in (1, 2, 3, 4, 5):
            raise ValueError(f"Invalid score: {score}")
        motivation = str(parsed.get("motivation", "")).strip()
        if not motivation:
            raise ValueError("Empty motivation")
        return {"te_score": score, "motivation": motivation}
    except Exception as e:
        return {"te_score": -1, "motivation": f"Parse error: {str(e)[:120]}"}


def query_gemini_te(occupation, stage_name, task):
    """Query Gemini 2.5 Flash-Lite for TE score."""
    full_prompt = SYSTEM_PROMPT_TE + "\n\n" + build_te_prompt(occupation, stage_name, task)
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            resp = gemini_model.generate_content(
                full_prompt,
                generation_config=genai.types.GenerationConfig(temperature=0, max_output_tokens=700)
            )
            result = parse_te_response(resp.text)
            if result["te_score"] != -1:
                return result
            if attempt < MAX_RETRIES:
                time.sleep(10)
        except Exception as e:
            err = str(e)
            if "429" in err or "quota" in err.lower():
                time.sleep(60 + attempt * 15)
            elif attempt < MAX_RETRIES:
                time.sleep(10)
    return {"te_score": -1, "motivation": "Gemini: all attempts failed"}


def query_mistral_te(occupation, stage_name, task):
    """Query Mistral Small 3.2 for TE score."""
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            resp = mistral_client.chat.complete(
                model=MISTRAL_MODEL,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT_TE},
                    {"role": "user",   "content": build_te_prompt(occupation, stage_name, task)},
                ],
                temperature=0,
                max_tokens=700,
            )
            result = parse_te_response(resp.choices[0].message.content)
            if result["te_score"] != -1:
                return result
            if attempt < MAX_RETRIES:
                time.sleep(10)
        except Exception as e:
            err = str(e)
            if "429" in err or "rate" in err.lower():
                time.sleep(60 + attempt * 15)
            elif attempt < MAX_RETRIES:
                time.sleep(10)
    return {"te_score": -1, "motivation": "Mistral: all attempts failed"}


def query_deepseek_te(occupation, stage_name, task):
    """Query DeepSeek V3 for TE score."""
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            resp = deepseek_client.chat.completions.create(
                model=DEEPSEEK_MODEL,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT_TE},
                    {"role": "user",   "content": build_te_prompt(occupation, stage_name, task)},
                ],
                temperature=0,
                max_tokens=700,
            )
            result = parse_te_response(resp.choices[0].message.content)
            if result["te_score"] != -1:
                return result
            if attempt < MAX_RETRIES:
                time.sleep(10)
        except Exception as e:
            err = str(e)
            if "429" in err or "rate" in err.lower():
                time.sleep(60 + attempt * 15)
            elif attempt < MAX_RETRIES:
                time.sleep(10)
    return {"te_score": -1, "motivation": "DeepSeek: all attempts failed"}


print("All query functions ready.")


In [ ]:
smoke_tests = [
    ("Biostatisticians",
     "Stage 3 — Clinical Research",
     "Calculate sample size requirements for clinical studies using statistical power and significance thresholds."),
    ("Veterinary Assistants and Laboratory Animal Caretakers",
     "Stage 2 — Preclinical Research",
     "Anesthetize and inoculate animals according to study protocol instructions."),
]

print("=" * 72)
for occ, stage, task in smoke_tests:
    print(f"\nTask : {task[:70]}...")
    print(f"Stage: {stage}")
    g = query_gemini_te(occ, stage, task)
    m = query_mistral_te(occ, stage, task)
    d = query_deepseek_te(occ, stage, task)
    valid = [x["te_score"] for x in [g, m, d] if x["te_score"] != -1]
    avg = round(sum(valid) / len(valid), 2) if valid else "N/A"
    print(f"  Gemini   [{g['te_score']}]: {g['motivation'][:90]}...")
    print(f"  Mistral  [{m['te_score']}]: {m['motivation'][:90]}...")
    print(f"  DeepSeek [{d['te_score']}]: {d['motivation'][:90]}...")
    print(f"  --> TE Score (avg): {avg}")
    print("-" * 72)
    time.sleep(1)

print("\nSmoke test complete. If scores look plausible, proceed to full run.")


In [ ]:
df = pd.read_excel(INPUT_FILE)
df["Task ID"] = df["Task ID"].astype(str)
df["stage_label"] = df["dominant_stage"].map({
    1: "Stage 1 — Discovery & Development",
    2: "Stage 2 — Preclinical Research",
    3: "Stage 3 — Clinical Research",
    4: "Stage 4 — Regulatory Review",
})

print(f"Geladen : {len(df):,} Tasks aus {df['Title'].nunique()} Occupations")
for s, n in STAGE_NAMES.items():
    c = (df["dominant_stage"] == s).sum()
    print(f"  {n}: {c} tasks")
df.head(3)


In [ ]:

if os.path.exists(CHECKPOINT_CSV):
    done_df  = pd.read_csv(CHECKPOINT_CSV, dtype={"Task ID": str})
    done_ids = set(done_df["Task ID"].tolist())
    print(f"Resume: {len(done_ids):,} tasks already evaluated.")
else:
    done_df  = pd.DataFrame()
    done_ids = set()
    print("Starting fresh.")

remaining = df[~df["Task ID"].isin(done_ids)].copy()
print(f"Remaining: {len(remaining):,} tasks")
print(f"Running 3 models IN PARALLEL per task (ThreadPoolExecutor)\n")

def evaluate_task(row):
    """
    Query all 3 models concurrently for a single task.
    Returns a result dict ready for the checkpoint buffer.
    """
    occ       = str(row["Title"])
    task_text = str(row["Task"])
    stage_lbl = str(row["stage_label"])
    task_id   = str(row["Task ID"])


    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
        fut_g = ex.submit(query_gemini_te,   occ, stage_lbl, task_text)
        fut_m = ex.submit(query_mistral_te,  occ, stage_lbl, task_text)
        fut_d = ex.submit(query_deepseek_te, occ, stage_lbl, task_text)
        g = fut_g.result()
        m = fut_m.result()
        d = fut_d.result()

    valid_scores = [x["te_score"] for x in [g, m, d] if x["te_score"] != -1]
    te_score = round(sum(valid_scores) / len(valid_scores), 3) if valid_scores else -1

    return {
        "Task ID":             task_id,
        "te_gemini":           g["te_score"],
        "motivation_gemini":   g["motivation"],
        "te_mistral":          m["te_score"],
        "motivation_mistral":  m["motivation"],
        "te_deepseek":         d["te_score"],
        "motivation_deepseek": d["motivation"],
        "te_score":            te_score,
        "n_valid_models":      len(valid_scores),
    }

buffer = []

for _, row in tqdm(remaining.iterrows(), total=len(remaining), desc="TE Scoring"):
    result = evaluate_task(row)
    buffer.append(result)

    if len(buffer) % 25 == 0:
        snap = pd.concat([done_df, pd.DataFrame(buffer)], ignore_index=True)
        snap.drop_duplicates(subset="Task ID", keep="last").to_csv(CHECKPOINT_CSV, index=False)
        print(f"  Checkpoint: {len(snap):,} tasks saved.")

final_te = pd.concat([done_df, pd.DataFrame(buffer)], ignore_index=True) if buffer else done_df
final_te.drop_duplicates(subset="Task ID", keep="last").to_csv(CHECKPOINT_CSV, index=False)
print(f"\nDone! {len(final_te):,} tasks evaluated.")
print(f"Errors (te_score = -1): {(final_te['te_score'] == -1).sum()}")


In [ ]:

te_df = pd.read_csv(CHECKPOINT_CSV, dtype={"Task ID": str})


merged = df.merge(
    te_df[["Task ID",
           "te_gemini",   "motivation_gemini",
           "te_mistral",  "motivation_mistral",
           "te_deepseek", "motivation_deepseek",
           "te_score",    "n_valid_models"]],
    on="Task ID", how="left"
)


task_cols = [
    "O*NET-SOC Code", "Title", "Task ID", "Task", "Task Type",
    "dominant_stage", "stage_label",
    "te_gemini",  "te_mistral",  "te_deepseek",  "te_score",
    "motivation_gemini", "motivation_mistral", "motivation_deepseek",
    "n_valid_models",
    "D_final", "D_gemini", "D_deepseek", "D_mistral32",
]
task_cols = [c for c in task_cols if c in merged.columns]
task_out = merged[task_cols].sort_values(["dominant_stage", "Title"]).reset_index(drop=True)


stage_rows = []
sn_map = {1:"Discovery & Development", 2:"Preclinical Research",
          3:"Clinical Research", 4:"Regulatory Review"}
for s, sname in sn_map.items():
    sub = task_out[task_out["dominant_stage"] == s]
    v   = sub[sub["te_score"] > 0]
    stage_rows.append({
        "stage_number":     s,
        "stage_name":       sname,
        "n_tasks":          len(sub),
        "te_mean":          round(v["te_score"].mean(), 3) if len(v) else None,
        "te_median":        round(v["te_score"].median(), 3) if len(v) else None,
        "te_min":           round(v["te_score"].min(), 2) if len(v) else None,
        "te_max":           round(v["te_score"].max(), 2) if len(v) else None,
        "te_gemini_mean":   round(v["te_gemini"].mean(), 3) if len(v) else None,
        "te_mistral_mean":  round(v["te_mistral"].mean(), 3) if len(v) else None,
        "te_deepseek_mean": round(v["te_deepseek"].mean(), 3) if len(v) else None,
        "pct_high_4_5":     round(100 * (v["te_score"] >= 4).mean(), 1) if len(v) else None,
        "pct_low_1_2":      round(100 * (v["te_score"] <= 2).mean(), 1) if len(v) else None,
    })

stage_out = pd.DataFrame(stage_rows)

all_v = task_out[task_out["te_score"] > 0]
print("=" * 60)
print("TE SCORE OVERVIEW")
print("=" * 60)
print(f"Total tasks   : {len(task_out)}")
print(f"Valid scores  : {len(all_v)}")
print(f"Overall mean  : {all_v['te_score'].mean():.3f} / 5.0")
print(f"Overall median: {all_v['te_score'].median():.3f}")
print()
for s, sname in sn_map.items():
    sub = all_v[all_v["dominant_stage"] == s]
    pct = 100 * (sub["te_score"] >= 4).mean() if len(sub) > 0 else 0
    print(f"Stage {s} — {sname:<30}: mean={sub['te_score'].mean():.3f}  n={len(sub)}  high(≥4)={pct:.1f}%")
print()
print(stage_out.to_string(index=False))


In [ ]:
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter
from openpyxl import load_workbook

STAGE_COLORS = {1: "D6E4F0", 2: "D5F5E3", 3: "FEF9E7", 4: "FADBD8"}
TE_COLORS    = {5: "1A7A4A", 4: "52BE80", 3: "F0B429", 2: "E8735A", 1: "C0392B"}

def style_task_sheet(ws):
    for cell in ws[1]:
        cell.font      = Font(bold=True, color="FFFFFF", name="Arial", size=10)
        cell.fill      = PatternFill("solid", fgColor="2E4057")
        cell.alignment = Alignment(horizontal="center", wrap_text=True)
    headers = [c.value for c in ws[1]]
    s_idx   = headers.index("dominant_stage") + 1 if "dominant_stage" in headers else None
    te_idx  = headers.index("te_score")       + 1 if "te_score"       in headers else None
    g_idx   = headers.index("te_gemini")      + 1 if "te_gemini"      in headers else None
    m_idx   = headers.index("te_mistral")     + 1 if "te_mistral"     in headers else None
    d_idx   = headers.index("te_deepseek")    + 1 if "te_deepseek"    in headers else None
    for row in ws.iter_rows(min_row=2):
        sv = row[s_idx-1].value if s_idx else None
        bg = STAGE_COLORS.get(sv, "FFFFFF")
        for cell in row:
            cell.fill      = PatternFill("solid", fgColor=bg)
            cell.font      = Font(name="Arial", size=9)
            cell.alignment = Alignment(wrap_text=True, vertical="top")
        for idx in [te_idx, g_idx, m_idx, d_idx]:
            if idx:
                sc = row[idx-1]
                rv = round(sc.value) if sc.value and isinstance(sc.value, (int, float)) and sc.value > 0 else None
                if rv and rv in TE_COLORS:
                    sc.fill = PatternFill("solid", fgColor=TE_COLORS[rv])
                    sc.font = Font(name="Arial", size=9, bold=True, color="FFFFFF")
    for i, col in enumerate(ws.columns, 1):
        h = ws.cell(1, i).value or ""
        if h == "Task":
            ws.column_dimensions[get_column_letter(i)].width = 55
        elif "motivation" in str(h):
            ws.column_dimensions[get_column_letter(i)].width = 60
        elif h in ("Title", "stage_label"):
            ws.column_dimensions[get_column_letter(i)].width = 28
        elif str(h).startswith("te_"):
            ws.column_dimensions[get_column_letter(i)].width = 12
        else:
            ws.column_dimensions[get_column_letter(i)].width = 14
    ws.freeze_panes = "A2"

def style_stage_sheet(ws):
    for cell in ws[1]:
        cell.font      = Font(bold=True, color="FFFFFF", name="Arial", size=10)
        cell.fill      = PatternFill("solid", fgColor="2E4057")
        cell.alignment = Alignment(horizontal="center", wrap_text=True)
    headers = [c.value for c in ws[1]]
    s_idx = headers.index("stage_number") + 1 if "stage_number" in headers else None
    for row in ws.iter_rows(min_row=2):
        sv = row[s_idx-1].value if s_idx else None
        bg = STAGE_COLORS.get(sv, "F8F9FA")
        for cell in row:
            cell.fill      = PatternFill("solid", fgColor=bg)
            cell.font      = Font(name="Arial", size=10, bold=True)
            cell.alignment = Alignment(horizontal="center", wrap_text=True)
    for i in range(1, ws.max_column + 1):
        ws.column_dimensions[get_column_letter(i)].width = 18
    ws.freeze_panes = "A2"

with pd.ExcelWriter(OUTPUT_EXCEL, engine="openpyxl") as writer:
    task_out.to_excel(writer,  sheet_name="Task_TE_Scores",   index=False)
    stage_out.to_excel(writer, sheet_name="Stage_TE_Summary", index=False)

wb = load_workbook(OUTPUT_EXCEL)
style_task_sheet(wb["Task_TE_Scores"])
style_stage_sheet(wb["Stage_TE_Summary"])
wb.save(OUTPUT_EXCEL)

print(f"Saved: {OUTPUT_EXCEL}")
print(f"  Task_TE_Scores   : {len(task_out):,} rows")
print(f"  Stage_TE_Summary : {len(stage_out)} rows")


In [ ]:
sn_map = {1:"Discovery & Development", 2:"Preclinical Research",
          3:"Clinical Research", 4:"Regulatory Review"}
all_v = task_out[task_out["te_score"] > 0]

print("=" * 65)
print("DDL GENERATIVE AI EXPOSURE — TE SCORE SUMMARY")
print("Following Colombo et al. (2026) — adapted for GenAI scope")
print("=" * 65)
print(f"Tasks evaluated     : {len(task_out)}")
print(f"Valid scores        : {len(all_v)}")
print(f"Overall mean TE     : {all_v['te_score'].mean():.3f} / 5.000")
print(f"Overall median TE   : {all_v['te_score'].median():.3f}")
print()
print(f"{'Stage':<35} {'n':>4} {'Mean':>6} {'Med':>6} {'≥4(%)':>7} {'≤2(%)':>7}")
print("-" * 65)
for s, sname in sn_map.items():
    sub = all_v[all_v["dominant_stage"] == s]
    if len(sub) == 0:
        continue
    p_high = 100 * (sub["te_score"] >= 4).mean()
    p_low  = 100 * (sub["te_score"] <= 2).mean()
    print(f"Stage {s} — {sname:<28} {len(sub):>4} "
          f"{sub['te_score'].mean():>6.3f} {sub['te_score'].median():>6.3f} "
          f"{p_high:>6.1f}% {p_low:>6.1f}%")
print("-" * 65)
p_high_all = 100 * (all_v["te_score"] >= 4).mean()
p_low_all  = 100 * (all_v["te_score"] <= 2).mean()
print(f"{'Overall':<35} {len(all_v):>4} "
      f"{all_v['te_score'].mean():>6.3f} {all_v['te_score'].median():>6.3f} "
      f"{p_high_all:>6.1f}% {p_low_all:>6.1f}%")
print()
print("Model-level mean TE scores:")
print(f"  Gemini 2.5 Flash-Lite  : {all_v['te_gemini'].mean():.3f}")
print(f"  Mistral Small 3.2      : {all_v['te_mistral'].mean():.3f}")
print(f"  DeepSeek V3            : {all_v['te_deepseek'].mean():.3f}")
